# Sequence Retrieval with the PBI Package

This notebook demonstrates all the ways to retrieve sequences and data from the PBI package, from simple metadata queries to full sequence retrieval and batch iteration.

## What You'll Learn
1. **Database Connection** - How `quick_connect()` works internally
2. **Database Statistics** - Overview of available data
3. **Querying Phage Metadata** - SQL-style filtering
4. **Querying Host Metadata** - Filter by assembly quality, species
5. **Querying Phage-Host Pairs** - Metadata-only joins
6. **Pagination** - LIMIT and OFFSET for large datasets
7. **Retrieving Phage Sequences** - DNA sequences from FASTA
8. **Retrieving Host Sequences** - Bacterial genome sequences
9. **Retrieving Protein Sequences** - Phage protein sequences
10. **Getting DataFrames** - Tabular results with sequences
11. **Batch Iteration** - Memory-efficient processing
12. **Understanding Warnings** - Missing sequence messages explained

## How PBI Works Internally

```
pbi.quick_connect()
  |
  +-- Finds database file (auto-detected from common paths)
  +-- Opens DuckDB connection (fast, in-process SQL)
  +-- Locates FASTA files (phage, protein, host sequences)
  +-- Builds pyfaidx indexes if not present (.fai files)
  |
  v
SequenceRetriever
  |
  +-- get_phage_metadata()     -> SQL query -> DataFrame
  +-- get_host_metadata()      -> SQL query -> DataFrame
  +-- get_phage_host_pairs()   -> SQL + FASTA lookup -> DataFrame with sequences
  +-- create_streaming_dataset() -> Iterator over samples
  +-- create_indexed_dataset()   -> Random-access dataset
```

## Setup & Imports

In [ ]:
import sys
from pathlib import Path
import time

project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import pbi
import pandas as pd

# Set pandas display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("PBI package imported successfully")
print(f"Version: {pbi.__version__ if hasattr(pbi, '__version__') else 'dev'}")

## Database Connection

`pbi.quick_connect()` searches for the PBI database and FASTA files automatically. It:
1. **Looks in common locations** for the `.duckdb` database file
2. **Opens a DuckDB connection** — DuckDB is an embedded database (like SQLite but faster for analytics)
3. **Locates FASTA files** for phage sequences, protein sequences, and host sequences
4. **Builds FASTA indexes** (`.fai` files) using pyfaidx if they don't exist yet

The connection is nearly instant because FASTA files are indexed — sequences are retrieved with direct byte-offset reads, not by scanning the entire file.

In [ ]:
# Quick connect - auto-detects database and FASTA files
print("Connecting to PBI database...")
start = time.time()
retriever = pbi.quick_connect()
elapsed = time.time() - start

print(f"Connected in {elapsed:.2f} seconds")
print(f"Database connection: {retriever.conn}")
print(f"Phage FASTA ready: {retriever._phage_fasta is not None}")
print(f"Host data available: {retriever._has_host_data if hasattr(retriever, '_has_host_data') else 'unknown'}")

## Database Statistics

The `get_stats()` method runs a few COUNT queries and checks FASTA file sizes. It returns a nested dictionary with:
- `stats['database']`: Record counts from DuckDB tables
- `stats['fasta']`: Sequence counts from indexed FASTA files

The difference between database and FASTA counts reflects the coverage — not all database records have associated sequences.

In [ ]:
# Get comprehensive statistics
stats = retriever.get_stats()

print("Database Statistics:")
print("=" * 60)
print(f"Phages:   {stats['database']['phages']:,}")
print(f"Proteins: {stats['database']['proteins']:,}")
print(f"Hosts:    {stats['database']['hosts']:,}")
print(f"Phage-Host Associations: {stats['database']['phage_host_associations']:,}")
print()
print("FASTA Files:")
print(f"  Phage sequences:   {stats['fasta']['phages']:,}")
print(f"  Protein sequences: {stats['fasta']['proteins']:,}")
if 'hosts' in stats['fasta']:
    print(f"  Host sequences:    {stats['fasta']['hosts']:,}")

## Querying Phage Metadata

`retriever.get_phage_metadata()` executes a SELECT query against the `fact_phages` table. You can pass any SQL WHERE clause (without the `WHERE` keyword), or use `LIMIT`/`OFFSET` directly.

**Available columns typically include:**
- `Phage_ID`: Unique identifier
- `Length`: Genome length in base pairs
- `GC_content`: GC percentage
- `Taxonomy`: Taxonomic classification
- `Completeness`: Genome completeness status
- `Lifestyle`: Lytic, lysogenic, or unknown
- `Source_DB`: Which database the record came from
- `Host`: Recorded host organism

In [ ]:
# Get all phage metadata (no filter)
phage_metadata = retriever.get_phage_metadata()
print(f"Total phages: {len(phage_metadata):,}")
print(f"Columns: {list(phage_metadata.columns)}")
phage_metadata.head()

In [ ]:
# Filter: only complete phages
try:
    complete_phages = retriever.get_phage_metadata(
        where_clause="Completeness = 'complete'"
    )
    print(f"Complete phages: {len(complete_phages):,}")
    print(set(complete_phages['Completeness']))
except Exception:
    # Try with different capitalization
    completeness_values = phage_metadata['Completeness'].value_counts()
    print("Available completeness values:")
    print(completeness_values.head(10))

In [ ]:
# Filter: large phages (>100 kb)
large_phages = retriever.get_phage_metadata(
    where_clause="Length > 100000"
)
print(f"Large phages (>100 kb): {len(large_phages):,}")
if len(large_phages) > 0:
    print(f"Largest: {large_phages['Length'].max():,} bp")
    print(large_phages[['Phage_ID', 'Length', 'GC_content', 'Taxonomy']].head())

## Querying Host Metadata

`retriever.get_host_metadata()` queries the `dim_hosts` table. Host records represent bacterial genomes with assembly information from NCBI.

**Available columns typically include:**
- `Host_ID`: Unique identifier (usually NCBI assembly accession)
- `Species_Name`: Bacterial species
- `Assembly_Level`: Complete Genome, Scaffold, Contig, etc.
- `Genome_Length`: Total genome size in bp
- `GC_Content`: Host GC percentage

In [ ]:
# Get host metadata with filter
try:
    host_metadata = retriever.get_host_metadata(
        where_clause="Assembly_Level = 'Complete Genome'",
        limit=10
    )
    print(f"Complete genome hosts (sample): {len(host_metadata)}")
    print(host_metadata[['Species_Name', 'Assembly_Level', 'Genome_Length', 'GC_Content']].head())
except Exception as e:
    print(f"Host metadata query: {e}")
    # Try without filter
    host_metadata = retriever.get_host_metadata()
    print(f"All hosts: {len(host_metadata):,}")
    print(f"Assembly levels: {host_metadata['Assembly_Level'].value_counts().head()}")

## Querying Phage-Host Pairs (Metadata Only)

`retriever.get_phage_host_metadata()` (or `get_phage_host_pairs(sequences=False)`) performs a SQL JOIN between phage and host tables through the association table. This returns metadata without loading sequences — fast for large queries.

**Use this when you need:**
- Statistics about pairs
- Filtering before sequence retrieval
- Building feature tables from metadata alone

In [ ]:
# Query phage-host pairs (metadata only, no sequences)
try:
    pairs_meta = retriever.get_phage_host_metadata(
        where_clause="Length > 10000 AND Species_Name LIKE 'Escherichia%'",
        limit=20
    )
    print(f"Escherichia-infecting phages (>10kb): {len(pairs_meta)}")
    if len(pairs_meta) > 0:
        print(pairs_meta[['Phage_ID', 'Phage_Length', 'Host_Species', 'Host_Assembly_Level']].head())
except Exception as e:
    print(f"Note: {e}")
    pairs_meta = retriever.get_phage_host_metadata(limit=5)
    print(f"Sample pairs (no filter): {len(pairs_meta)}")
    if len(pairs_meta) > 0:
        print(pairs_meta.head())

## LIMIT and OFFSET for Pagination

For large datasets, use LIMIT and OFFSET to process data in pages. This is more memory-efficient than loading everything at once.

The `where_clause` parameter accepts these directly at the end of the clause:

In [ ]:
# Pagination with LIMIT and OFFSET
print("Pagination example:")
print("-" * 60)

# First 5 phages
batch1 = retriever.get_phage_metadata(where_clause="LIMIT 5")
print(f"Batch 1 (first 5): {list(batch1['Phage_ID'])}")

# Next 5 phages
batch2 = retriever.get_phage_metadata(where_clause="LIMIT 5 OFFSET 5")
print(f"Batch 2 (next 5): {list(batch2['Phage_ID'])}")

# Verify they don't overlap
overlap = set(batch1['Phage_ID']) & set(batch2['Phage_ID'])
print(f"Overlap between batches: {len(overlap)} (expected 0)")

## Retrieving Phage Sequences

`retriever.get_phage_host_pairs()` is the main method for retrieving sequences. It:
1. Runs a SQL JOIN to get metadata
2. For each row, looks up the phage sequence in the FASTA file using the `.fai` index
3. Looks up the host sequence in the host FASTA files
4. Returns a DataFrame with both metadata and sequences

**Under the hood (pyfaidx):** The `.fai` index stores byte offsets for each sequence. Retrieval is O(1) — it seeks directly to the right position without reading the entire file.

**Note:** Pairs where phage or host sequences are not found are automatically skipped with a warning. This is expected behavior.

In [ ]:
# Get phage-host pairs with sequences
print("Retrieving phage-host pairs with sequences...")
start = time.time()

pairs = retriever.get_phage_host_pairs(
    where_clause="LIMIT 5"
)

elapsed = time.time() - start
print(f"Retrieved {len(pairs)} pairs in {elapsed:.2f}s")
print(f"\nColumns: {list(pairs.columns)}")

if len(pairs) > 0:
    print(f"\nSample data:")
    print(pairs[['Phage_ID', 'Species_Name', 'Phage_Length', 'Host_Length']].head())

    print(f"\nSequence preview (first pair):")
    row = pairs.iloc[0]
    phage_seq = row.get('Phage_Sequence', '')
    host_seq = row.get('Host_Sequence', '')
    if phage_seq:
        print(f"  Phage {row['Phage_ID'][:30]}: {phage_seq[:60]}...")
    if host_seq:
        print(f"  Host {str(row.get('Species_Name', ''))[:30]}: {host_seq[:60]}...")

In [ ]:
# Filter by taxonomy for specific phage families
print("Retrieving Siphoviridae phage-host pairs...")
sipho_pairs = retriever.get_phage_host_pairs(
    where_clause="Taxonomy LIKE '%Siphoviridae%' LIMIT 10"
)
print(f"Found {len(sipho_pairs)} Siphoviridae phage-host pairs")
if len(sipho_pairs) > 0:
    print("\nTaxonomy distribution:")
    print(sipho_pairs['Phage_Taxonomy'].value_counts() if 'Phage_Taxonomy' in sipho_pairs.columns else sipho_pairs['Taxonomy'].value_counts())

## Retrieving Host Sequences

Host sequences are stored either in a combined FASTA file or as individual per-genome files (depending on the pipeline configuration). The `SequenceRetriever` handles both modes transparently.

When you call `get_phage_host_pairs()`, host sequences are retrieved automatically. You can also access the host FASTA directly via the retriever's internal attributes.

In [ ]:
# Inspect host sequence availability
print("Host sequence information:")
print("-" * 60)

if len(pairs) > 0:
    has_host_seq = pairs['Host_Sequence'].notna() & (pairs['Host_Sequence'] != '')
    print(f"Pairs with host sequences: {has_host_seq.sum()}/{len(pairs)}")
    if has_host_seq.any():
        sample = pairs[has_host_seq].iloc[0]
        host_seq = sample['Host_Sequence']
        print(f"\nSample host sequence:")
        print(f"  Species: {sample.get('Species_Name', 'N/A')}")
        print(f"  Length: {len(host_seq):,} bp")
        print(f"  First 80 bp: {host_seq[:80]}")
else:
    print("No pairs retrieved - check database content")

## Retrieving Protein Sequences

PBI stores phage protein sequences in a separate FASTA file. Proteins are annotated phage genes — multiple proteins per phage.

You can access protein sequences directly or filter proteins by phage ID using SQL.

In [ ]:
# Query protein metadata
try:
    proteins = retriever.get_protein_metadata(
        where_clause="LIMIT 5"
    )
    print(f"Sample protein records: {len(proteins)}")
    if len(proteins) > 0:
        print(f"Columns: {list(proteins.columns)}")
        print(proteins.head())
except AttributeError:
    # Fallback: query directly
    try:
        proteins_df = retriever.conn.execute(
            "SELECT * FROM dim_proteins LIMIT 5"
        ).fetchdf()
        print(f"Sample proteins from dim_proteins: {len(proteins_df)}")
        print(proteins_df.head())
    except Exception as e2:
        print(f"Protein table query: {e2}")
except Exception as e:
    print(f"Protein query: {e}")

## Getting Results as DataFrames

All retrieval methods return pandas DataFrames. This makes it easy to:
- Export to CSV/Parquet
- Merge with other datasets
- Apply pandas operations (groupby, pivot, merge)
- Feed into sklearn or other ML libraries

In [ ]:
# Example: analyze sequence lengths from retrieved pairs
if len(pairs) > 0:
    print("Sequence length analysis:")
    print("-" * 60)

    if 'Phage_Length' in pairs.columns:
        print(f"Phage lengths: min={pairs['Phage_Length'].min():,}, max={pairs['Phage_Length'].max():,}, mean={pairs['Phage_Length'].mean():,.0f}")

    if 'Host_Length' in pairs.columns:
        host_len = pairs['Host_Length'].dropna()
        if len(host_len) > 0:
            print(f"Host lengths: min={host_len.min():,}, max={host_len.max():,}, mean={host_len.mean():,.0f}")

    # Export example
    # pairs.to_csv('phage_host_pairs.csv', index=False)
    # pairs.to_parquet('phage_host_pairs.parquet', index=False)
    print("\nTo export: pairs.to_csv('output.csv', index=False)")

## Batch Iteration with `get_phage_host_pairs_iterator()`

For datasets too large to load all at once, use the batch iterator. It:
- Runs paginated SQL queries internally
- Yields one DataFrame batch at a time
- Fetches sequences for each batch before moving to the next
- Keeps memory constant regardless of total dataset size

**When to use:**
- Processing >10,000 pairs
- Computing statistics over the full database
- Writing results to disk incrementally

In [ ]:
# Batch iteration example
print("Batch iteration example:")
print("-" * 60)

batch_iterator = retriever.get_phage_host_pairs_iterator(
    where_clause="p.Length > 10000",
    batch_size=100
)

max_batches = 3
total_processed = 0
all_phage_lengths = []

for i, batch_df in enumerate(batch_iterator):
    if i >= max_batches:
        break

    batch_size = len(batch_df)
    total_processed += batch_size
    if 'Phage_Length' in batch_df.columns:
        all_phage_lengths.extend(batch_df['Phage_Length'].dropna().tolist())

    print(f"Batch {i+1}: {batch_size} pairs")
    if i == 0:
        print(f"  Columns: {list(batch_df.columns[:6])}...")

print(f"\nTotal processed: {total_processed} pairs")
if all_phage_lengths:
    import numpy as np
    print(f"Mean phage length: {np.mean(all_phage_lengths):,.0f} bp")

## Understanding Missing Sequences (The Warnings Explained)

When retrieving sequences, you may see log messages like:

```
WARNING - Host genome not found in mapping: GCF_000123456.1
WARNING - Phage sequence not in FASTA: PHAGE_XYZ_001  
WARNING - Skipping pair: host genome not available
```

**These are informational, not errors.** The PBI pipeline:
1. Downloads phage sequences from source databases
2. Downloads host genomes from NCBI based on phage metadata
3. Stores them in indexed FASTA files

But not every phage has a sequenced host, and not every host genome was successfully downloaded. The retriever skips unavailable sequences and continues.

**To track which pairs are skipped**, use the `missing_hosts_csv` parameter:

In [ ]:
# Track missing sequences during retrieval
import tempfile
import os

missing_csv = Path(tempfile.mktemp(suffix='_missing.csv'))

try:
    dataset = retriever.create_indexed_dataset(
        where_clause="LIMIT 50",
        missing_hosts_csv=str(missing_csv)
    )
    print(f"Created dataset with {len(dataset)} samples")

    # Access a few samples (triggers sequence lookup)
    for i in range(min(5, len(dataset))):
        _ = dataset[i]

    if missing_csv.exists():
        missing_df = pd.read_csv(missing_csv)
        print(f"Missing sequences logged: {len(missing_df)}")
        if len(missing_df) > 0:
            print("\nFailure reasons:")
            print(missing_df['Failure_Reason'].value_counts())
    else:
        print("All sequences found - no missing data")
except Exception as e:
    print(f"Note: {e}")
finally:
    if missing_csv.exists():
        os.unlink(str(missing_csv))

## Summary

This notebook demonstrated all sequence retrieval methods in PBI:

- **`get_phage_metadata()`** - SQL-filtered phage metadata
- **`get_host_metadata()`** - SQL-filtered host metadata
- **`get_phage_host_metadata()`** - Metadata-only pair queries (fast)
- **`get_phage_host_pairs()`** - Pairs with DNA sequences (slower, loads FASTA)
- **`get_phage_host_pairs_iterator()`** - Batch iteration for large datasets
- **`create_streaming_dataset()`** / **`create_indexed_dataset()`** - ML-ready datasets

### Key Architecture Points
- DuckDB provides fast SQL queries on metadata
- pyfaidx provides O(1) sequence access via `.fai` index files
- Missing sequences are skipped gracefully with warnings
- All methods return pandas DataFrames (except iterator and datasets)

See `03_ml_streaming.ipynb` for machine learning applications.

In [ ]:
# Cleanup
retriever.close()
print("Database connection closed")